# 04. Add database-derived 192-component spectrum weights

This notebook adds population database weights to the 192-component spectrum-group table produced by `03_prepare_spectrum_groups_192.ipynb`.

The weight columns themselves are used only for mutation spectrum construction. However, population AF features derived from the same gnomAD and Helix sources are used by the upstream Isolation Forest, so the weighted spectrum is descriptive rather than an independent validation of the model groups.

For each possible SNV, raw counts are taken from gnomAD and Helix and then combined. A floor of one is applied after combining the two databases (zero becomes one; positive counts are unchanged):

`combined_db_count_pc = max(gnomad_count_raw + helix_count_raw, 1)`

The spectrum weight is then normalized by the combined database size and by the number of occurrences of the exact reference trinucleotide in circular chrM:

`combined_db_spectrum_weight = combined_db_count_pc / (combined_db_total_records * ref_context_count)`

Here `ref_context_count` is counted once per genomic position, not once per possible alternate allele. The resulting table is saved as `results/spectrum_groups/spectrum_groups_with_weights_192_T95.tsv`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

In [2]:
# ============================================================
# Paths
# ============================================================

GROUPS_INPUT = Path("../results/spectrum_groups/spectrum_groups_192_T95.tsv")
MASTER_INPUT = Path("../data/processed/master_snv_table.tsv")

OUTPUT_DIR = Path("../results/spectrum_groups")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTED_GROUPS_OUTPUT = OUTPUT_DIR / "spectrum_groups_with_weights_192_T95.tsv"
WEIGHT_QC_OUTPUT = OUTPUT_DIR / "spectrum_weight_qc_192_T95.tsv"

In [3]:
# ============================================================
# Load input tables
# ============================================================

groups = pd.read_csv(GROUPS_INPUT, sep="\t", low_memory=False)
master = pd.read_csv(MASTER_INPUT, sep="\t", low_memory=False)

print("Groups shape:", groups.shape)
print("Master shape:", master.shape)

display(groups.head())
display(master.head())

Groups shape: (49704, 34)
Master shape: (49704, 64)


,variant_id,position,reference,alternate,left_base,right_base,trinucleotide_context,substitution_type_12,substitution_type_192,is_valid_snv_substitution,validation_label,is_neutral_dataset8,is_pathogenic_dataset9,is_disease_suspected_dataset3,analysis_group,isolation_forest_outlier_score,isolation_forest_above_T95,mlc_score,pop_af_max,pop_af_hom_max,pop_af_het_max,rarity_soft,hom_rarity_soft,het_rarity_soft,no_homoplasmic_signal,position_mod3,codon_position_simple,codon_pos1_any,codon_pos2_any,codon_pos3_any,phyloP100way,spectrum_group_primary_T95,spectrum_group_strict_T95,spectrum_group_posthoc
0,m.1G>A,1,G,A,G,A,GGA,G>A,G[G>A]A,1,unlabeled,0,0,0,unlabeled_or_other,0.106907,1,0.65755,0.0,0.0,0.0,6.0,6.0,6.0,1,NaN,noncoding,0,0,0,0.109654,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95
1,m.1G>C,1,G,C,G,A,GGA,G>C,G[G>C]A,1,unlabeled,0,0,0,unlabeled_or_other,0.106907,1,0.65755,0.0,0.0,0.0,6.0,6.0,6.0,1,NaN,noncoding,0,0,0,0.109654,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95
2,m.1G>T,1,G,T,G,A,GGA,G>T,G[G>T]A,1,unlabeled,0,0,0,unlabeled_or_other,0.106907,1,0.65755,0.0,0.0,0.0,6.0,6.0,6.0,1,NaN,noncoding,0,0,0,0.109654,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95
3,m.2A>C,2,A,C,G,T,GAT,A>C,G[A>C]T,1,unlabeled,0,0,0,unlabeled_or_other,0.105476,1,0.64832,0.0,0.0,0.0,6.0,6.0,6.0,1,NaN,noncoding,0,0,0,0.109654,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95
4,m.2A>G,2,A,G,G,T,GAT,A>G,G[A>G]T,1,unlabeled,0,0,0,unlabeled_or_other,0.105476,1,0.64832,0.0,0.0,0.0,6.0,6.0,6.0,1,NaN,noncoding,0,0,0,0.109654,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95


,position,reference,alternate,consequence,mlc_score,variant_id,mlc_position_score,gnomad_observed,gnomad_filter,AN,gnomad_homoplasmic_ac,gnomad_heteroplasmic_ac,gnomad_homoplasmic_af,gnomad_heteroplasmic_af,gnomad_combined_af_simple,gnomad_max_heteroplasmy,hap_defining_variant,vep,feature,gene,helix_counts_hom,helix_af_hom,helix_counts_het,helix_af_het,helix_mean_arf,helix_max_arf,helix_haplogroups_hom,helix_haplogroups_het,helix_observed,is_disease_suspected_dataset3,is_neutral_dataset8,is_pathogenic_dataset9,validation_label,gene_constraint_symbol,gene_constraint_start_position,gene_constraint_end_position,gene_constraint_consequence,gene_constraint_observed,gene_constraint_expected,gene_constraint_obs_exp,gene_constraint_lower_ci,gene_constraint_upper_ci,regional_constraint_symbol,regional_constraint_start_position,regional_constraint_end_position,regional_constraint_protein_residue_start,regional_constraint_protein_residue_end,regional_constraint_observed,regional_constraint_expected,regional_constraint_obs_exp,regional_constraint_lower_ci,regional_constraint_upper_ci,in_regional_constraint,noncoding_constraint_locus,noncoding_constraint_description,noncoding_constraint_start_position,noncoding_constraint_end_position,noncoding_constraint_observed,noncoding_constraint_expected,noncoding_constraint_obs_exp,noncoding_constraint_lower_ci,noncoding_constraint_upper_ci,in_noncoding_constraint,is_artifact_prone_site
0,1,G,A,intergenic_variant,0.65755,m.1G>A,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,1,G,C,intergenic_variant,0.65755,m.1G>C,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,1,G,T,intergenic_variant,0.65755,m.1G>T,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,2,A,C,intergenic_variant,0.64832,m.2A>C,0.64832,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,2,A,G,intergenic_variant,0.64832,m.2A>G,0.64832,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [4]:
# ============================================================
# Required columns
# ============================================================

required_group_cols = [
    "variant_id",
    "position",
    "reference",
    "alternate",
    "left_base",
    "right_base",
    "trinucleotide_context",
    "substitution_type_192",
    "is_valid_snv_substitution",
    "spectrum_group_primary_T95",
    "spectrum_group_strict_T95",
    "spectrum_group_posthoc",
]

missing_group_cols = [col for col in required_group_cols if col not in groups.columns]
if missing_group_cols:
    raise ValueError(f"Missing required columns in groups table: {missing_group_cols}")

required_master_cols = [
    "variant_id",
    "AN",
    "gnomad_homoplasmic_ac",
    "gnomad_heteroplasmic_ac",
    "gnomad_observed",
    "helix_counts_hom",
    "helix_counts_het",
    "helix_af_hom",
    "helix_af_het",
]

missing_master_cols = [col for col in required_master_cols if col not in master.columns]
if missing_master_cols:
    raise ValueError(f"Missing required columns in master table: {missing_master_cols}")

print("All required columns are present.")

All required columns are present.


In [5]:
# ============================================================
# Trinucleotide-context opportunity counts
# ============================================================

n_invalid_context = int((groups["is_valid_snv_substitution"] != 1).sum())
work = groups[groups["is_valid_snv_substitution"] == 1].copy()
print("Excluded variants without an A/C/G/T trinucleotide context:", n_invalid_context)
work["reference"] = work["reference"].astype(str).str.upper()
work["alternate"] = work["alternate"].astype(str).str.upper()

valid_bases = {"A", "C", "G", "T"}
context_is_valid = (
    work["left_base"].isin(valid_bases)
    & work["reference"].isin(valid_bases)
    & work["right_base"].isin(valid_bases)
    & work["alternate"].isin(valid_bases)
    & work["alternate"].ne(work["reference"])
)
if not context_is_valid.all():
    raise ValueError(f"Invalid trinucleotide SNVs: {int((~context_is_valid).sum())}")

# Count genomic opportunities once per position. Each of the three alternate
# alleles at a position shares the same reference trinucleotide context.
context_positions = (
    work[["position", "trinucleotide_context"]]
    .drop_duplicates("position")
)
context_counts = context_positions["trinucleotide_context"].value_counts().to_dict()
work["ref_context_count"] = work["trinucleotide_context"].map(context_counts)

if work["ref_context_count"].isna().any() or (work["ref_context_count"] <= 0).any():
    raise ValueError("Could not assign positive ref_context_count values.")

print("Reference trinucleotide contexts:", len(context_counts))
print("Total mtDNA positions:", int(sum(context_counts.values())))
display(
    work[[
        "variant_id", "trinucleotide_context", "substitution_type_192",
        "ref_context_count",
    ]].head()
)

Excluded variants without an A/C/G/T trinucleotide context: 6
Reference trinucleotide contexts: 64
Total mtDNA positions: 16566


,variant_id,trinucleotide_context,substitution_type_192,ref_context_count
0,m.1G>A,GGA,G[G>A]A,123
1,m.1G>C,GGA,G[G>C]A,123
2,m.1G>T,GGA,G[G>T]A,123
3,m.2A>C,GAT,G[A>C]T,114
4,m.2A>G,GAT,G[A>G]T,114


In [6]:
# ============================================================
# Prepare population counts from master table
# ============================================================

def to_numeric_or_zero(df, col):
    return pd.to_numeric(df[col], errors="coerce").fillna(0.0)


master_pop = master[
    [
        "variant_id",
        "AN",
        "gnomad_observed",
        "gnomad_homoplasmic_ac",
        "gnomad_heteroplasmic_ac",
        "helix_counts_hom",
        "helix_counts_het",
        "helix_af_hom",
        "helix_af_het",
    ]
].copy()

master_pop = master_pop.drop_duplicates("variant_id")

# gnomAD raw count
master_pop["gnomad_count_raw"] = (
    to_numeric_or_zero(master_pop, "gnomad_homoplasmic_ac")
    + to_numeric_or_zero(master_pop, "gnomad_heteroplasmic_ac")
)

# Helix raw count
master_pop["helix_count_raw"] = (
    to_numeric_or_zero(master_pop, "helix_counts_hom")
    + to_numeric_or_zero(master_pop, "helix_counts_het")
)

# Raw combined count
master_pop["combined_db_count_raw"] = (
    master_pop["gnomad_count_raw"]
    + master_pop["helix_count_raw"]
)

# Observed flags
master_pop["gnomad_db_observed"] = (
    master_pop["gnomad_count_raw"] > 0
).astype(int)

master_pop["helix_db_observed"] = (
    master_pop["helix_count_raw"] > 0
).astype(int)

master_pop["combined_db_observed"] = (
    master_pop["combined_db_count_raw"] > 0
).astype(int)

master_pop["n_population_dbs_observed"] = (
    master_pop["gnomad_db_observed"]
    + master_pop["helix_db_observed"]
)

display(master_pop.head())

,variant_id,AN,gnomad_observed,gnomad_homoplasmic_ac,gnomad_heteroplasmic_ac,helix_counts_hom,helix_counts_het,helix_af_hom,helix_af_het,gnomad_count_raw,helix_count_raw,combined_db_count_raw,gnomad_db_observed,helix_db_observed,combined_db_observed,n_population_dbs_observed
0,m.1G>A,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0,0,0,0
1,m.1G>C,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0,0,0,0
2,m.1G>T,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0,0,0,0
3,m.2A>C,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0,0,0,0
4,m.2A>G,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0,0,0,0


In [7]:
# ============================================================
# Infer database denominators
# ============================================================

def infer_total_from_count_and_af(df, count_cols, af_cols):
    ratios = []

    for count_col, af_col in zip(count_cols, af_cols):
        count = pd.to_numeric(df[count_col], errors="coerce")
        af = pd.to_numeric(df[af_col], errors="coerce")

        mask = (
            count.notna()
            & af.notna()
            & (count > 0)
            & (af > 0)
        )

        if mask.any():
            ratios.append(count[mask] / af[mask])

    if not ratios:
        return np.nan

    ratios = pd.concat(ratios, axis=0)
    ratios = ratios.replace([np.inf, -np.inf], np.nan).dropna()

    if ratios.empty:
        return np.nan

    return int(round(ratios.median()))


# gnomAD denominator
gnomad_total_records = int(
    round(pd.to_numeric(master_pop["AN"], errors="coerce").dropna().max())
)

# Helix denominator
helix_total_records = infer_total_from_count_and_af(
    master_pop,
    count_cols=["helix_counts_hom", "helix_counts_het"],
    af_cols=["helix_af_hom", "helix_af_het"],
)

if pd.isna(gnomad_total_records) or gnomad_total_records <= 0:
    raise ValueError("Could not infer gnomAD total records from AN.")

if pd.isna(helix_total_records) or helix_total_records <= 0:
    raise ValueError(
        "Could not infer Helix total records from count / AF. "
        "Set helix_total_records manually."
    )

combined_db_total_records = int(gnomad_total_records + helix_total_records)

print("gnomAD total records:", gnomad_total_records)
print("Helix total records:", helix_total_records)
print("Combined total records:", combined_db_total_records)

gnomAD total records: 56434
Helix total records: 195983
Combined total records: 252417


In [8]:
# ============================================================
# Merge population counts into grouped spectrum table
# ============================================================

merge_cols = [
    "variant_id",
    "gnomad_count_raw",
    "helix_count_raw",
    "combined_db_count_raw",
    "gnomad_db_observed",
    "helix_db_observed",
    "combined_db_observed",
    "n_population_dbs_observed",
]

work = work.merge(
    master_pop[merge_cols],
    on="variant_id",
    how="left",
    validate="one_to_one",
)

count_cols = [
    "gnomad_count_raw",
    "helix_count_raw",
    "combined_db_count_raw",
]

flag_cols = [
    "gnomad_db_observed",
    "helix_db_observed",
    "combined_db_observed",
    "n_population_dbs_observed",
]

work[count_cols] = work[count_cols].fillna(0.0)
work[flag_cols] = work[flag_cols].fillna(0).astype(int)

print("After merge:", work.shape)
display(work.head())

After merge: (49698, 42)


,variant_id,position,reference,alternate,left_base,right_base,trinucleotide_context,substitution_type_12,substitution_type_192,is_valid_snv_substitution,validation_label,is_neutral_dataset8,is_pathogenic_dataset9,is_disease_suspected_dataset3,analysis_group,isolation_forest_outlier_score,isolation_forest_above_T95,mlc_score,pop_af_max,pop_af_hom_max,pop_af_het_max,rarity_soft,hom_rarity_soft,het_rarity_soft,no_homoplasmic_signal,position_mod3,codon_position_simple,codon_pos1_any,codon_pos2_any,codon_pos3_any,phyloP100way,spectrum_group_primary_T95,spectrum_group_strict_T95,spectrum_group_posthoc,ref_context_count,gnomad_count_raw,helix_count_raw,combined_db_count_raw,gnomad_db_observed,helix_db_observed,combined_db_observed,n_population_dbs_observed
0,m.1G>A,1,G,A,G,A,GGA,G>A,G[G>A]A,1,unlabeled,0,0,0,unlabeled_or_other,0.106907,1,0.65755,0.0,0.0,0.0,6.0,6.0,6.0,1,NaN,noncoding,0,0,0,0.109654,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,123,0.0,0.0,0.0,0,0,0,0
1,m.1G>C,1,G,C,G,A,GGA,G>C,G[G>C]A,1,unlabeled,0,0,0,unlabeled_or_other,0.106907,1,0.65755,0.0,0.0,0.0,6.0,6.0,6.0,1,NaN,noncoding,0,0,0,0.109654,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,123,0.0,0.0,0.0,0,0,0,0
2,m.1G>T,1,G,T,G,A,GGA,G>T,G[G>T]A,1,unlabeled,0,0,0,unlabeled_or_other,0.106907,1,0.65755,0.0,0.0,0.0,6.0,6.0,6.0,1,NaN,noncoding,0,0,0,0.109654,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,123,0.0,0.0,0.0,0,0,0,0
3,m.2A>C,2,A,C,G,T,GAT,A>C,G[A>C]T,1,unlabeled,0,0,0,unlabeled_or_other,0.105476,1,0.64832,0.0,0.0,0.0,6.0,6.0,6.0,1,NaN,noncoding,0,0,0,0.109654,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,114,0.0,0.0,0.0,0,0,0,0
4,m.2A>G,2,A,G,G,T,GAT,A>G,G[A>G]T,1,unlabeled,0,0,0,unlabeled_or_other,0.105476,1,0.64832,0.0,0.0,0.0,6.0,6.0,6.0,1,NaN,noncoding,0,0,0,0.109654,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,unlabeled_out_of_neutral_domain_T95,114,0.0,0.0,0.0,0,0,0,0


In [9]:
# ============================================================
# Add pseudocount-normalized 192-component spectrum weights
# ============================================================

def add_spectrum_weight(
    df,
    prefix,
    raw_count_col,
    total_records,
    opportunity_count_col="ref_context_count",
):
    raw_count = pd.to_numeric(df[raw_count_col], errors="coerce").fillna(0.0)
    opportunity_count = pd.to_numeric(df[opportunity_count_col], errors="coerce")

    if total_records <= 0:
        raise ValueError(f"Invalid total_records for {prefix}: {total_records}")
    if (opportunity_count <= 0).any():
        raise ValueError(f"Invalid opportunity count detected for {prefix}.")

    df[f"{prefix}_count_pc"] = np.maximum(raw_count, 1.0)
    df[f"{prefix}_freq_raw"] = raw_count / total_records
    df[f"{prefix}_freq_pc"] = df[f"{prefix}_count_pc"] / total_records

    # Normalize each variant by the number of occurrences of its reference
    # trinucleotide in the circular mitochondrial genome.
    df[f"{prefix}_spectrum_weight"] = (
        df[f"{prefix}_freq_pc"] / opportunity_count
    )
    df[f"{prefix}_spectrum_weight_raw"] = (
        df[f"{prefix}_freq_raw"] / opportunity_count
    )
    df[f"{prefix}_total_records"] = int(total_records)
    return df


work = add_spectrum_weight(
    work, "gnomad_db", "gnomad_count_raw", gnomad_total_records
)
work = add_spectrum_weight(
    work, "helix_db", "helix_count_raw", helix_total_records
)
work = add_spectrum_weight(
    work, "combined_db", "combined_db_count_raw", combined_db_total_records
)

In [10]:
# ============================================================
# Sanity checks
# ============================================================

weight_cols = [
    "gnomad_db_spectrum_weight",
    "helix_db_spectrum_weight",
    "combined_db_spectrum_weight",
]

for col in weight_cols:
    if work[col].isna().any():
        raise ValueError(f"NaN values detected in {col}")

    if (work[col] <= 0).any():
        raise ValueError(f"Non-positive values detected in {col}")

print("Observed variants:")
print("gnomAD:", int(work["gnomad_db_observed"].sum()))
print("Helix:", int(work["helix_db_observed"].sum()))
print("Combined:", int(work["combined_db_observed"].sum()))

print("\nWeight summary:")
display(work[weight_cols].describe())

print("\nTop variants by combined_db_spectrum_weight:")
display(
    work.sort_values("combined_db_spectrum_weight", ascending=False)
    [
        [
            "variant_id",
            "substitution_type_192",
            "gnomad_count_raw",
            "helix_count_raw",
            "combined_db_count_raw",
            "ref_context_count",
            "combined_db_spectrum_weight",
            "combined_db_observed",
            "n_population_dbs_observed",
        ]
    ]
    .head(20)
)

Observed variants:
gnomAD: 10440
Helix: 13409
Combined: 14501

Weight summary:


,gnomad_db_spectrum_weight,helix_db_spectrum_weight,combined_db_spectrum_weight
count,4.969800e+04,4.969800e+04,4.969800e+04
mean,2.701697e-06,2.046106e-06,2.180207e-06
std,5.905600e-05,5.397650e-05,5.467822e-05
min,2.839714e-08,8.177057e-09,6.348875e-09
25%,3.955316e-08,1.217776e-08,9.455127e-09
50%,5.469078e-08,1.656650e-08,1.290455e-08
75%,1.150637e-07,4.148360e-08,3.329158e-08
max,5.973082e-03,6.079753e-03,6.055904e-03



Top variants by combined_db_spectrum_weight:


,variant_id,substitution_type_192,gnomad_count_raw,helix_count_raw,combined_db_count_raw,ref_context_count,combined_db_spectrum_weight,combined_db_observed,n_population_dbs_observed
49546,m.16519T>C,G[T>C]C,35731.0,126302.0,162033.0,106,0.006056,1,2
2248,m.750A>G,G[A>G]A,55425.0,192999.0,248424.0,201,0.004896,1,2
35145,m.11719G>A,G[G>A]C,39885.0,115582.0,155467.0,151,0.004079,1,2
14296,m.4769A>G,T[A>G]G,55442.0,191449.0,246891.0,258,0.003791,1,2
4312,m.1438A>G,T[A>G]A,53749.0,189907.0,243656.0,414,0.002332,1,2
787,m.263A>G,C[A>G]C,55906.0,193872.0,249778.0,454,0.002180,1,2
26569,m.8860A>G,C[A>G]C,56078.0,193621.0,249699.0,454,0.002179,1,2
45967,m.15326A>G,A[A>G]C,56039.0,193990.0,250029.0,495,0.002001,1,2
217,m.73A>G,T[A>G]T,40205.0,119994.0,160199.0,324,0.001959,1,2
2124,m.709G>A,C[G>A]T,8220.0,28918.0,37138.0,78,0.001886,1,2


In [11]:
# ============================================================
# Compact QC table
# ============================================================

qc = pd.DataFrame({
    "metric": [
        "n_variants",
        "n_observed_gnomad",
        "n_observed_helix",
        "n_observed_combined",
        "gnomad_total_records",
        "helix_total_records",
        "combined_db_total_records",
    ],
    "value": [
        len(work),
        int(work["gnomad_db_observed"].sum()),
        int(work["helix_db_observed"].sum()),
        int(work["combined_db_observed"].sum()),
        int(gnomad_total_records),
        int(helix_total_records),
        int(combined_db_total_records),
    ],
})

qc.to_csv(WEIGHT_QC_OUTPUT, sep="\t", index=False)
display(qc)
print("Saved QC:", WEIGHT_QC_OUTPUT)


,metric,value
0,n_variants,49698
1,n_observed_gnomad,10440
2,n_observed_helix,13409
3,n_observed_combined,14501
4,gnomad_total_records,56434
5,helix_total_records,195983
6,combined_db_total_records,252417


Saved QC: ..\results\spectrum_groups\spectrum_weight_qc_192_T95.tsv


In [12]:
# ============================================================
# Save weighted grouped table
# ============================================================

new_weight_cols = [
    "ref_context_count",

    "gnomad_count_raw",
    "gnomad_db_count_pc",
    "gnomad_db_freq_raw",
    "gnomad_db_freq_pc",
    "gnomad_db_spectrum_weight_raw",
    "gnomad_db_spectrum_weight",
    "gnomad_db_total_records",
    "gnomad_db_observed",

    "helix_count_raw",
    "helix_db_count_pc",
    "helix_db_freq_raw",
    "helix_db_freq_pc",
    "helix_db_spectrum_weight_raw",
    "helix_db_spectrum_weight",
    "helix_db_total_records",
    "helix_db_observed",

    "combined_db_count_raw",
    "combined_db_count_pc",
    "combined_db_freq_raw",
    "combined_db_freq_pc",
    "combined_db_spectrum_weight_raw",
    "combined_db_spectrum_weight",
    "combined_db_total_records",
    "combined_db_observed",

    "n_population_dbs_observed",
]

for col in new_weight_cols:
    if col not in work.columns:
        raise ValueError(f"Missing new output column: {col}")

work.to_csv(WEIGHTED_GROUPS_OUTPUT, sep="\t", index=False)

print("Saved weighted grouped table:")
print(WEIGHTED_GROUPS_OUTPUT)

print("\nFinal shape:", work.shape)

Saved weighted grouped table:
..\results\spectrum_groups\spectrum_groups_with_weights_192_T95.tsv

Final shape: (49698, 60)
